In [36]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'key': ['A', 'B', 'C', 'A', 'A', 'B'], 
    'value': [1, 2, 3, 4, 5, 6],
    'value2': [10, 20, 30, 40, 50, 60]
})

group = df.groupby('key')

print(group)

for key, group in group:
    display(group)

,key,value,value2
0,A,1,10
3,A,4,40
4,A,5,50


,key,value,value2
1,B,2,20
5,B,6,60


,key,value,value2
2,C,3,30


In [42]:
df_g = pd.DataFrame({
    'group': ['A', 'A', 'B', 'B', 'C', 'C'],
    'value': [100, 200, 300, 400, np.nan, np.nan],
    'key': ['X', 'Y', 'X', 'Y', 'X', 'Y']
})

df_g['key'] = df_g['key'].astype('category') # cast na category

# dodamy nową "pustą" kategorię 'Z'
df_g['key'] = df_g['key'].cat.add_categories(['Z']) 


In [38]:
print("Kategorie w kolumnie 'key':", df_g['key'].cat.categories)

Kategorie w kolumnie 'key': Index(['X', 'Y', 'Z'], dtype='object')


Idempotentna wersja dodawania kategorii

In [ ]:
# df_g['key'] = (
#     df_g['key'].cat.add_categories(['Z']) if not 'Z' in df_g['key'].dtype.categories else print('Category exist.')
# )

Category exist.


In [40]:
display(df_g)

,group,value,key
0,A,100.0,None
1,A,200.0,None
2,B,300.0,None
3,B,400.0,None
4,C,NaN,None
5,C,NaN,None


In [ ]:
# display(
#     df_g.groupby('key', observed=True)['value'].sum()
# )

Series([], Name: value, dtype: float64)

In [43]:
display(df_g.groupby('key', observed=False).apply(pd.DataFrame)) 

C:\Users\singl\AppData\Local\Temp\ipykernel_17700\603555723.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  display(df_g.groupby('key', observed=False).apply(pd.DataFrame))


group  value key
key                   
X   0     A  100.0   X
    2     B  300.0   X
    4     C    NaN   X
Y   1     A  200.0   Y
    3     B  400.0   Y
    5     C    NaN   Y

In [ ]:
import numpy as np
import pandas as pd

df_g = pd.DataFrame({
    'group': ['A', 'A', 'B', 'B', 'C', 'C'],
    'value': [100, 200, 300, 400, np.nan, np.nan],
    'key': ['X', 'Y', 'X', 'Y', 'X', 'Y']
})

df_g['key'] = df_g['key'].astype('category')   


df_g['key'] = df_g['key'].cat.add_categories(['Z']) 

display(df_g.groupby('key', observed=False).apply(pd.DataFrame))  

C:\Users\singl\AppData\Local\Temp\ipykernel_17700\2747352342.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  display(df_g.groupby('key', observed=False).apply(pd.DataFrame))


group  value key
key                   
X   0     A  100.0   X
    2     B  300.0   X
    4     C    NaN   X
Y   1     A  200.0   Y
    3     B  400.0   Y
    5     C    NaN   Y

In [53]:
display(df_g)

,group,value,key
0,A,100.0,X
1,A,200.0,Y
2,B,300.0,X
3,B,400.0,Y
4,C,NaN,X
5,C,NaN,Y


In [62]:
print(df_g.index.names)
print(df_g.columns.tolist())

[None]
['group', 'value', 'key']


In [59]:
grouped = (
    df_g.groupby(
        by=['group', 'key'], # grupowanie po wielu kluczach
        observed=False
    )
    .apply(
        pd.DataFrame, 
        include_groups=False
    )
)
display(grouped)

value
group key         
A     X   0  100.0
      Y   1  200.0
B     X   2  300.0
      Y   3  400.0
C     X   4    NaN
      Y   5    NaN

In [61]:
print(grouped.index.names)
print(grouped.columns.tolist())

['group', 'key', None]
['value']


In [68]:
# display(grouped.swaplevel())
display(grouped.swaplevel())
display(grouped.swaplevel(1, 2))
display(grouped.swaplevel(0, 1))
display(grouped.swaplevel(0, 2))

value
group   key       
A     0 X    100.0
      1 Y    200.0
B     2 X    300.0
      3 Y    400.0
C     4 X      NaN
      5 Y      NaN

value
group   key       
A     0 X    100.0
      1 Y    200.0
B     2 X    300.0
      3 Y    400.0
C     4 X      NaN
      5 Y      NaN

,,,value
key,group,,
X,A,0,100.0
Y,A,1,200.0
X,B,2,300.0
Y,B,3,400.0
X,C,4,NaN
Y,C,5,NaN


,,,value
,key,group,
0,X,A,100.0
1,Y,A,200.0
2,X,B,300.0
3,Y,B,400.0
4,X,C,NaN
5,Y,C,NaN


---
---

#### `pivot_table()`

- index:   lista lub pojedyncza kolumna przeniesiona do indexu
- columns: lista lub pojedyncza kolumna
- values:  wartości do zagregowania
- aggfunc: funckja agregująca

In [70]:
df_sprzedaz = pd.DataFrame({
    'sklep': ['A', 'A', 'B', 'B'],
    'miesiac': ['Styczen', 'Luty', 'Styczen', 'Luty'],
    'produkt': ['X', 'Y', 'X', 'X'],
    'kwota': [100, 150, 200, 250],
    'tax': [10, 15, 20, 25],
    'ilosc': [1, 2, 3, 4]
})

display(df_sprzedaz)

,sklep,miesiac,produkt,kwota,tax,ilosc
0,A,Styczen,X,100,10,1
1,A,Luty,Y,150,15,2
2,B,Styczen,X,200,20,3
3,B,Luty,X,250,25,4


In [72]:
wynik = df_sprzedaz.pivot_table(
    index='sklep', # może być też lista ['something']
    columns=['miesiac', 'produkt'],
    values=['kwota', 'ilosc'],
    aggfunc='sum'
).fillna(0)

display(wynik)

ilosc               kwota               
miesiac  Luty      Styczen   Luty        Styczen
produkt     X    Y       X      X      Y       X
sklep                                           
A         0.0  2.0     1.0    0.0  150.0   100.0
B         4.0  0.0     3.0  250.0    0.0   200.0

In [78]:
display(
    wynik.swaplevel(
        axis=1,
        # j=-2, i=-1 # domyślne zachwoanie metody swaplevel
        j=0, i=1 # domyślne zachwoanie metody swaplevel
    ) # axis=1 odnosi się do MultiIndexu column
)

miesiac  Luty      Styczen   Luty        Styczen
        ilosc        ilosc  kwota          kwota
produkt     X    Y       X      X      Y       X
sklep                                           
A         0.0  2.0     1.0    0.0  150.0   100.0
B         4.0  0.0     3.0  250.0    0.0   200.0

In [80]:
display(f"Ilość poziomów MultiIndeksu w kolumnach: {wynik.columns.nlevels}")
display(f"Ilość poziomów MultiIndeksu w indeksie: {wynik.index.nlevels}")

'Ilość poziomów MultiIndeksu w kolumnach: 3'

'Ilość poziomów MultiIndeksu w indeksie: 1'

Spłaszczanie indeksów

In [87]:
wynik_ = wynik.copy()

print(wynik_.columns)
print(wynik_.columns.values)

# Spłaszczanie
flattened_cols = ['_'.join(col).strip().lower() for col in wynik_.columns.values]
print(flattened_cols)

wynik_.columns = flattened_cols

wynik_ = wynik_.reset_index(drop=False)

display(wynik_)




MultiIndex([('ilosc',    'Luty', 'X'),
            ('ilosc',    'Luty', 'Y'),
            ('ilosc', 'Styczen', 'X'),
            ('kwota',    'Luty', 'X'),
            ('kwota',    'Luty', 'Y'),
            ('kwota', 'Styczen', 'X')],
           names=[None, 'miesiac', 'produkt'])
[('ilosc', 'Luty', 'X') ('ilosc', 'Luty', 'Y') ('ilosc', 'Styczen', 'X')
 ('kwota', 'Luty', 'X') ('kwota', 'Luty', 'Y') ('kwota', 'Styczen', 'X')]
['ilosc_luty_x', 'ilosc_luty_y', 'ilosc_styczen_x', 'kwota_luty_x', 'kwota_luty_y', 'kwota_styczen_x']


,sklep,ilosc_luty_x,ilosc_luty_y,ilosc_styczen_x,kwota_luty_x,kwota_luty_y,kwota_styczen_x
0,A,0.0,2.0,1.0,0.0,150.0,100.0
1,B,4.0,0.0,3.0,250.0,0.0,200.0


In [93]:
import os
display(wynik)

sink = 'data'
os.makedirs(sink, exist_ok=True)


wynik.to_csv(sink+"/wynik_.csv", index=False)
wynik_.to_csv(sink+"/wynik.csv", index=False)

ilosc               kwota               
miesiac  Luty      Styczen   Luty        Styczen
produkt     X    Y       X      X      Y       X
sklep                                           
A         0.0  2.0     1.0    0.0  150.0   100.0
B         4.0  0.0     3.0  250.0    0.0   200.0

Zapis MultiIndex DataFrame do Parquet

In [ ]:
wynik.to_parquet('data/wynik_multiindex.parquet', index=True)

In [95]:
wynik_read_parquet = pd.read_parquet('data/wynik_multiindex.parquet')
display(wynik_read_parquet)

ilosc               kwota               
miesiac  Luty      Styczen   Luty        Styczen
produkt     X    Y       X      X      Y       X
sklep                                           
A         0.0  2.0     1.0    0.0  150.0   100.0
B         4.0  0.0     3.0  250.0    0.0   200.0